# EXP-001 multi-doc 하락 진단

목적: EXP-001(hybrid)에서 multi-doc Recall@5가 떨어진 원인 가설 두 개를 데이터로 가른다(PR #46 논의).

- 가설 A: multi-doc 질의는 정답 본문과 어휘가 덜 겹쳐 BM25가 약하다
- 가설 B: 질의가 회의를 가리키는 말(연도·날짜·위원회·위원 이름·법안)이 본문에 없고 메타데이터에만 있다

판정이 아닌 관찰이다. 질의 원문은 출력하지 않고 집계만 낸다(D-10).

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from kiwipiepy import Kiwi

from rag.bm25 import forms
from rag.eval import metrics as m1
from rag.eval_multi import metrics as mm
from rag.index import load_cfg
from rag.search import read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/runs"
DENSE = RUNS / "kure-v1-fixed-512-64-multidoc.csv"
HYBRID = RUNS / "kure-v1-fixed-512-64-hybrid-multidoc.csv"
K = 5
CFG = load_cfg(["+exp=exp001", "index.scope=full"])
TAGS = set(CFG.retriever.bm25.tags)

In [3]:
def top_k(path, k):
    run = pd.read_csv(path, dtype={"qid": str, "doc_id": str}).sort_values(["qid", "rank"])
    return run.groupby("qid")["doc_id"].agg(lambda s: list(s)[:k])


def squash(s):
    return re.sub(r"\s+", "", s)


def core_committee(name):
    """'농림…위원회-예산결산심사소위원회' → '농림…' (위원회 앞부분)."""
    return re.sub(r"(특별)?위원회$", "", name.split("-")[0])

## 데이터

In [4]:
gold = mm.load_gold(ROOT / CFG.paths.multidoc_frozen)
with (ROOT / CFG.paths.multidoc_frozen).open(encoding="utf-8") as f:
    frozen = pd.DataFrame([json.loads(line) for line in f]).set_index("qid").loc[gold.index]
with (ROOT / CFG.paths.multidoc_pools).open(encoding="utf-8") as f:
    pool_key = {p["pool_id"]: p["key"] for p in map(json.loads, f)}
gold["pool"] = frozen["pool_doc_ids"].map(set)
gold["key"] = gold["pool_id"].map(pool_key)
dense5, hybrid5 = top_k(DENSE, K)[gold.index], top_k(HYBRID, K)[gold.index]
gold.shape, gold["type"].value_counts().to_dict()

((600, 5), {'conf': 233, 'questioner': 215, 'law': 152})

In [5]:
meta_cols = ["conference_number", "date", "committee_name", "meeting_name"]
need = set().union(*gold["gold"], *dense5, *hybrid5)
meta, text = {}, {}
with (ROOT / CFG.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        meta[d["doc_id"]] = {c: d[c] for c in meta_cols}
        if d["doc_id"] in need:
            text[d["doc_id"]] = d["context"]
meta = pd.DataFrame.from_dict(meta, orient="index")
len(meta), len(text)

(38516, 4000)

## 1. 유형별 상위 5위 변화

dense에서 정답이 하나 이상 5위 안에 있다가 hybrid에서 하나도 없어진 질의(이탈)와 그 반대(진입).

In [6]:
hit_d = [bool(g & set(d)) for g, d in zip(gold["gold"], dense5, strict=True)]
hit_h = [bool(g & set(h)) for g, h in zip(gold["gold"], hybrid5, strict=True)]
labels = {(1, 0): "이탈", (0, 1): "진입", (1, 1): "유지", (0, 0): "둘 다 실패"}
gold["change"] = [labels[(a, b)] for a, b in zip(hit_d, hit_h, strict=True)]
pd.crosstab(gold["type"], gold["change"], margins=True)

change,둘 다 실패,유지,이탈,진입,All
type,,,,,
conf,2,224,5,2,233
law,66,49,29,8,152
questioner,137,48,16,14,215
All,205,321,50,24,600


## 2. hybrid가 새로 올린 문서의 정체

hybrid 상위 5에는 있고 dense 상위 5에는 없는 문서(올라온 문서)와, 반대로 빠진 문서를 정답과의 관계로 나눈다.

- 정답 / 묶음 안 비정답(후보 집합 안에서 정답이 아닌 문서) / 정답과 같은 회의 / 정답과 같은 위원회(다른 회의) / 그 밖

In [7]:
ORDER = ["정답", "묶음 안 비정답", "정답과 같은 회의", "정답과 같은 위원회", "그 밖"]


def relation(doc, q):
    if doc in q["gold"]:
        return "정답"
    if doc in q["pool"]:
        return "묶음 안 비정답"
    confs = {meta.at[g, "conference_number"] for g in q["gold"]}
    comms = {meta.at[g, "committee_name"] for g in q["gold"]}
    if meta.at[doc, "conference_number"] in confs:
        return "정답과 같은 회의"
    if meta.at[doc, "committee_name"] in comms:
        return "정답과 같은 위원회"
    return "그 밖"


rows = []
for qid, q in gold.iterrows():
    d, h = dense5[qid], hybrid5[qid]
    base = {"type": q["type"], "change": q["change"]}
    rows += [{**base, "side": "올라온 문서", "rel": relation(x, q)} for x in h if x not in d]
    rows += [{**base, "side": "빠진 문서", "rel": relation(x, q)} for x in d if x not in h]
moved = pd.DataFrame(rows)
pd.crosstab([moved["type"], moved["side"]], moved["rel"]).reindex(columns=ORDER, fill_value=0)

rel                정답  묶음 안 비정답  정답과 같은 회의  정답과 같은 위원회  그 밖
type       side                                            
conf       빠진 문서   31        15          0         105  258
           올라온 문서  19        25          0          90  275
law        빠진 문서   46        17          3          94  317
           올라온 문서  14         3          2          84  374
questioner 빠진 문서   26        32         24         125  531
           올라온 문서  22        10         21         156  529

### 이탈 질의만

In [8]:
lost = moved[moved["change"] == "이탈"]
pd.crosstab([lost["type"], lost["side"]], lost["rel"]).reindex(columns=ORDER, fill_value=0)

rel                정답  묶음 안 비정답  정답과 같은 회의  정답과 같은 위원회  그 밖
type       side                                            
conf       빠진 문서    6         0          0           3    7
           올라온 문서   0         0          0           4   12
law        빠진 문서   33         6          0          17   48
           올라온 문서   0         1          1          16   86
questioner 빠진 문서   17         4          3           8   30
           올라온 문서   0         1          1          17   43

### law: 문서 본문에 그 법안 이름이 있는가

법안 이름은 후보 집합 키(`law` 라벨)의 앞 낱말(예: '방송법')만 쓴다.

In [9]:
def law_in(doc, key):
    return squash(key.split()[0]) in squash(text[doc])


rows = []
for qid, q in gold[gold["type"] == "law"].iterrows():
    d, h = dense5[qid], hybrid5[qid]
    rows += [{"group": "정답 문서", "has_law": law_in(g, q["key"])} for g in q["gold"]]
    rows += [{"group": "올라온 문서", "has_law": law_in(x, q["key"])} for x in h if x not in d]
    rows += [{"group": "빠진 문서", "has_law": law_in(x, q["key"])} for x in d if x not in h]
pd.DataFrame(rows).groupby("group")["has_law"].agg(["size", "mean"]).round(3)

,size,mean
group,,
빠진 문서,477,0.262
올라온 문서,477,0.155
정답 문서,384,0.336


## 3. 질의와 정답 본문의 어휘 겹침 (가설 A)

BM25와 같은 Kiwi 태그로 질의와 정답 문서를 토큰화해, 질의의 서로 다른 형태 중 정답 본문에 나오는 비율을 본다. single-doc은 dev-full 질의와 그 정답 문서다. multi-doc은 정답 문서마다 따로 잰 비율의 평균(문서별)과 정답 문서 합집합 기준 비율(합집합)을 함께 낸다.

In [10]:
kiwi = Kiwi()
sd_gold = m1.load_gold(CFG, "dev-full")
sd_text = read_texts(ROOT / CFG.paths.queries, sd_gold.index.tolist())
keep = set(sd_gold["doc_id"]) - set(text)
with (ROOT / CFG.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["doc_id"] in keep:
            text[d["doc_id"]] = d["context"]
doc_ids = sorted(set(sd_gold["doc_id"]) | set().union(*gold["gold"]))
doc_tok = dict(zip(doc_ids, map(set, forms(kiwi, (text[d] for d in doc_ids), TAGS)), strict=True))
len(sd_text), len(doc_ids)

(3016, 4330)

In [11]:
def overlap(q, docs):
    q = set(q)
    if not q:
        return np.nan, np.nan
    per_doc = np.mean([len(q & doc_tok[d]) / len(q) for d in docs])
    union = len(q & set().union(*(doc_tok[d] for d in docs))) / len(q)
    return per_doc, union


cols = ["문서별", "합집합"]
sd = pd.DataFrame(
    [overlap(q, [d]) for q, d in zip(forms(kiwi, sd_text, TAGS), sd_gold["doc_id"], strict=True)],
    columns=cols,
).assign(type="single-doc")
md_ = pd.DataFrame(
    [overlap(q, g) for q, g in zip(forms(kiwi, frozen["query"], TAGS), gold["gold"], strict=True)],
    columns=cols,
).assign(type=gold["type"].to_numpy())
pd.concat([sd, md_]).groupby("type")[cols].agg(["mean", "median"]).round(3)

문서별           합집합       
             mean median   mean median
type                                  
conf        0.430  0.433  0.608  0.622
law         0.294  0.257  0.500  0.482
questioner  0.312  0.283  0.489  0.500
single-doc  0.764  0.778  0.764  0.778

## 4. 회의를 가리키는 말이 어디에 있는가 (가설 B)

질의에서 정규식으로 뽑은 표현이 정답 문서 메타데이터와 본문 중 어디에 있는지 센다. 값은 질의 비율이고, "본문" 항목은 그 표현을 언급한 질의 안에서의 비율이다.

- 연도: `YYYY년`. 정답 `date`의 연도와 같은지(연도=메타), 정답 본문에 있는지
- 날짜: `M월 D일`. 정답 본문에 있는지
- 위원회: 정답 `committee_name` 앞부분이 질의에 있는지, 본문에 있는지
- 이름: `○○○ 위원·의원·장관·차관`의 앞 2\~4자. 모든 정답 본문 / 하나 이상의 정답 본문에 있는지
- 법안(law): 후보 집합 키 앞 낱말이 질의에 있는지, 모든 정답 본문에 있는지

In [12]:
YEAR = re.compile(r"((?:19|20)\d{2})년")
DAY = re.compile(r"(\d{1,2})월\s?(\d{1,2})일")
NAME = re.compile(r"([가-힣]{2,4})\s?(?:위원|의원|장관|차관)")


def mentions(qtext, q):
    bodies = [squash(text[g]) for g in q["gold"]]
    years = {meta.at[g, "date"][:4] for g in q["gold"]}
    comms = {core_committee(meta.at[g, "committee_name"]) for g in q["gold"]}
    qs = squash(qtext)
    out = {}
    ys = set(YEAR.findall(qtext))
    out["연도 언급"] = bool(ys)
    out["연도=메타"] = bool(ys & years) if ys else np.nan
    out["연도 본문"] = any(y in b for y in ys for b in bodies) if ys else np.nan
    ds = {f"{m}월{d}일" for m, d in DAY.findall(qtext)}
    out["날짜 언급"] = bool(ds)
    out["날짜 본문"] = any(x in b for x in ds for b in bodies) if ds else np.nan
    cm = {c for c in comms if c and c in qs}
    out["위원회 언급"] = bool(cm)
    out["위원회 본문"] = any(c in b for c in cm for b in bodies) if cm else np.nan
    ns = set(NAME.findall(qtext))
    out["이름 언급"] = bool(ns)
    out["이름 본문(모든 정답)"] = (
        np.mean([all(n in b for b in bodies) for n in ns]) if ns else np.nan
    )
    out["이름 본문(정답 하나 이상)"] = (
        np.mean([any(n in b for b in bodies) for n in ns]) if ns else np.nan
    )
    if q["type"] == "law":
        lw = squash(q["key"].split()[0])
        out["법안 언급"] = lw in qs
        out["법안 본문(모든 정답)"] = all(lw in b for b in bodies)
    return out


ment = pd.DataFrame(
    [mentions(t, q) for t, (_, q) in zip(frozen["query"], gold.iterrows(), strict=True)],
    index=gold.index,
)
ment.astype(float).groupby(gold["type"]).mean().T.round(3)

type,conf,law,questioner
연도 언급,1.000,1.000,1.000
연도=메타,1.000,1.000,1.000
연도 본문,0.137,0.125,0.200
날짜 언급,1.000,1.000,0.986
날짜 본문,0.026,0.033,0.009
위원회 언급,0.987,0.987,0.963
위원회 본문,0.148,0.227,0.169
이름 언급,1.000,1.000,1.000
이름 본문(모든 정답),0.091,0.027,0.082
이름 본문(정답 하나 이상),0.248,0.185,0.299


### 이탈 질의와 그 밖의 질의 비교 (law·questioner)

In [13]:
sub = gold["type"].isin(["law", "questioner"])
lost_flag = (gold.loc[sub, "change"] == "이탈").rename("이탈")
ment[sub].astype(float).groupby([gold.loc[sub, "type"], lost_flag]).mean().T.round(3)

type               law        questioner       
이탈               False  True       False  True 
연도 언급            1.000  1.000      1.000  1.000
연도=메타            1.000  1.000      1.000  1.000
연도 본문            0.138  0.069      0.196  0.250
날짜 언급            1.000  1.000      0.990  0.938
날짜 본문            0.033  0.034      0.010  0.000
위원회 언급           0.992  0.966      0.965  0.938
위원회 본문           0.221  0.250      0.146  0.467
이름 언급            1.000  1.000      1.000  1.000
이름 본문(모든 정답)     0.026  0.029      0.074  0.188
이름 본문(정답 하나 이상)  0.187  0.175      0.272  0.635
법안 언급            0.862  1.000        NaN    NaN
법안 본문(모든 정답)     0.130  0.034        NaN    NaN

## 5. single-doc 질의에서의 같은 표현 (판정 질의)

메타데이터를 검색에 넣는 실험도 판정은 single-doc dev-full로 한다. 같은 표현이 `summary_q`에 얼마나 나오는지 본다.

In [14]:
sd_ment = pd.DataFrame(
    [
        mentions(t, {"gold": {d}, "type": "single"})
        for t, d in zip(sd_text, sd_gold["doc_id"], strict=True)
    ],
    index=sd_gold.index,
)
pd.concat(
    {
        "single-doc": sd_ment.astype(float).mean(),
        "multi-doc": ment.drop(columns=["법안 언급", "법안 본문(모든 정답)"]).astype(float).mean(),
    },
    axis=1,
).round(3)

,single-doc,multi-doc
연도 언급,0.010,1.000
연도=메타,0.226,1.000
연도 본문,1.000,0.157
날짜 언급,0.003,0.995
날짜 본문,1.000,0.022
위원회 언급,0.030,0.978
위원회 본문,0.955,0.175
이름 언급,0.069,1.000
이름 본문(모든 정답),0.903,0.072
이름 본문(정답 하나 이상),0.903,0.250


In [15]:
sd_hit = (
    m1.gold_ranks(m1.read_run(RUNS / "kure-v1-fixed-512-64-hybrid-dev-full.csv"), sd_gold, 10) <= 5
)
any_meta = sd_ment[["연도 언급", "날짜 언급", "위원회 언급", "이름 언급"]].any(axis=1)
pd.DataFrame({"n": any_meta.value_counts(), "hybrid R@5": sd_hit.groupby(any_meta).mean()}).round(4)

,n,hybrid R@5
False,2702,0.9156
True,314,0.9268
